In [ ]:
import sys
import os
import time
import yaml
import cloudpickle
sys.path.append(os.path.abspath('..'))
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import pybullet as p

STAGE_TO_WATCH = 0

print(f"TRUE LIVE Tracker initialized. Monitoring current raw state of Stage {STAGE_TO_WATCH}...")

config_path = os.path.abspath(os.path.join('..', 'configs', 'teacher_ppo.yaml'))
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

stage_key = f"stage_{STAGE_TO_WATCH}"
stage_config = config['stages'][stage_key]
reward_weights = config['rewards']

NUM_OBS = stage_config['num_obstacles']
RAND_OBS = stage_config['randomize_obstacles']
RAND_COINS = stage_config['randomize_coins']
RUN_NAME = stage_config['run_name']

env_raw = RoomDroneEnv(gui=True, num_obstacles=NUM_OBS, randomize_obstacles=RAND_OBS, randomize_coins=RAND_COINS, reward_weights=reward_weights)
env_vec = DummyVecEnv([lambda e=env_raw: e])

model_path = os.path.abspath(os.path.join('..', 'models', RUN_NAME, 'latest_model'))
vecnorm_path = f"{model_path}_vecnormalize.pkl"
print(f"Waiting for latest live brain at: {model_path}.zip")

while not (os.path.exists(f"{model_path}.zip") and os.path.exists(vecnorm_path)):
    print("Waiting for the SaveLatestCallback to dump the first live memory...")
    time.sleep(5)

env = VecNormalize.load(vecnorm_path, env_vec)
env.training = False
env.norm_reward = False
model = PPO.load(model_path, env=env)

obs = env.reset()
p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

print(f"[{RUN_NAME}] TRUE LIVE Simulation started. You are watching the raw training state...")

while True:
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, dones, infos = env.step(action)
    time.sleep(1./240.)

    if dones[0]:
        time.sleep(0.5)
        try:
            # FIX: Update VecNormalize stats in-place instead of creating a new wrapper.
            # Old approach: VecNormalize.load() + env.reset() caused a double reset —
            # DummyVecEnv already auto-resets on done=True (1st world rebuild),
            # then env.reset() triggered a 2nd resetSimulation() immediately after.
            # New approach: load stats via cloudpickle, copy obs_rms/ret_rms in-place.
            # obs already holds the new episode's first observation from the auto-reset.
            with open(vecnorm_path, 'rb') as f:
                saved = cloudpickle.load(f)
            env.obs_rms = saved.obs_rms
            env.ret_rms = saved.ret_rms
            model = PPO.load(model_path, env=env)
        except Exception:
            pass

pybullet build time: Jan 29 2025 23:16:28


TRUE LIVE Tracker initialized. Monitoring current raw state of Stage 0...
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Mesa
GL_RENDERER=llvmpipe (LLVM 20.1.2, 256 bits)
GL_VERSION=4.5 (Core Profile) Mesa 25.2.8-0ubuntu0.24.04.1
GL_SHADING_LANGUAGE_VERSION=4.50
pthread_getconcurrency()=0
Version = 4.5 (Core Profile) Mesa 25.2.8-0ubuntu0.24.04.1
Vendor = Mesa
Renderer = llvmpipe (LLVM 20.1.2, 256 bits)
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
Waiting for latest live brain at: /home/ihamzt/cross-modal-drone/models/Stage_0_Awakening/latest_model.zip

/home/ihamzt/miniconda3/envs/cross_modal_rl/lib/python3.10/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


[Stage_0_Awakening] TRUE LIVE Simulation started. You are watching the raw training state...
